Question 1

In [ ]:
%%writefile device_query.cu
#include <iostream>
#include <cuda_runtime.h>

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("--- REPORT DATA ---\n");
        printf("1. Device Name: %s\n", prop.name);
        printf("2. Compute Capability: %d.%d\n", prop.major, prop.minor);
        printf("3. Max Block Dimensions: [%d, %d, %d]\n", prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        printf("4. Max Grid Dimensions: [%d, %d, %d]\n", prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);
        printf("5. Global Memory: %zu bytes\n", prop.totalGlobalMem);
        printf("6. Shared Memory per Block: %zu bytes\n", prop.sharedMemPerBlock);
        printf("7. Constant Memory: %zu bytes\n", prop.totalConstMem);
        printf("8. Warp Size: %d\n", prop.warpSize);
        // Double precision check
        if (prop.major >= 1) printf("9. Double Precision Support: Yes\n");
        else printf("9. Double Precision Support: No\n");
    }
    return 0;
}

Writing device_query.cu


In [ ]:
!nvcc device_query.cu -o device_query
!./device_query

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
--- REPORT DATA ---
1. Device Name: Tesla T4
2. Compute Capability: 7.5
3. Max Block Dimensions: [1024, 1024, 64]
4. Max Grid Dimensions: [2147483647, 65535, 65535]
5. Global Memory: 15637086208 bytes
6. Shared Memory per Block: 49152 bytes
7. Constant Memory: 65536 bytes
8. Warp Size: 32
9. Double Precision Support: Yes


Question 2

In [3]:
%%writefile array_sum.cu
#include <iostream>
#include <cuda_runtime.h>

// 7. The CUDA Kernel that computes the sum
// Each thread handles one index of the array
__global__ void arraySumKernel(float *a, float *b, float *result, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        result[i] = a[i] + b[i];
    }
}

int main() {
    int n = 1000; // Size of the array
    size_t size = n * sizeof(float);

    // Host memory (CPU)
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_res = (float*)malloc(size);

    // Initialize input arrays with some values
    for (int i = 0; i < n; i++) {
        h_a[i] = 1.0f;
        h_b[i] = 2.0f;
    }

    // 1. Allocate device memory (GPU)
    float *d_a, *d_b, *d_res;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_res, size);

    // 2. Copy host memory to device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // 3. Initialize thread block and kernel grid dimensions
    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    // 4. Invoke CUDA kernel
    arraySumKernel<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_res, n);

    // 5. Copy results from device to host
    cudaMemcpy(h_res, d_res, size, cudaMemcpyDeviceToHost);

    // Verify the first few results
    printf("Result Verification (First 5 elements):\n");
    for (int i = 0; i < 5; i++) {
        printf("%.1f + %.1f = %.1f\n", h_a[i], h_b[i], h_res[i]);
    }

    // 6. Free device memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_res);
    free(h_a);
    free(h_b);
    free(h_res);

    return 0;
}

Writing array_sum.cu


In [4]:
!nvcc array_sum.cu -o array_sum
!./array_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Result Verification (First 5 elements):
1.0 + 2.0 = 3.0
1.0 + 2.0 = 3.0
1.0 + 2.0 = 3.0
1.0 + 2.0 = 3.0
1.0 + 2.0 = 3.0


Question 3


In [1]:
%%writefile matrix_add.cu
#include <iostream>
#include <cuda_runtime.h>

// CUDA Kernel for Matrix Addition
__global__ void matrixAddKernel(int *A, int *B, int *C, int N) {
    // Calculate the row and column index for each thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        int idx = row * N + col; // Flatten 2D index to 1D
        C[idx] = A[idx] + B[idx];
    }
}

int main() {
    int N = 512; // Matrix size N x N
    size_t size = N * N * sizeof(int);

    // Host allocation
    int *h_A = (int*)malloc(size);
    int *h_B = (int*)malloc(size);
    int *h_C = (int*)malloc(size);

    for (int i = 0; i < N * N; i++) {
        h_A[i] = 1; h_B[i] = 2;
    }

    // Device allocation
    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Define 2D block and grid dimensions
    dim3 threadsPerBlock(16, 16); // 16x16 = 256 threads
    dim3 blocksPerGrid((N + 15) / 16, (N + 15) / 16);

    matrixAddKernel<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaMemcpy(h_C, h_C, size, cudaMemcpyDeviceToHost);

    printf("Top-left element: %d + %d = %d\n", h_A[0], h_B[0], h_A[0] + h_B[0]);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

Writing matrix_add.cu


In [2]:
!nvcc matrix_add.cu -o matrix_add
!./matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Top-left element: 1 + 2 = 3
